# Survey (DMKD, B6 — QĐ176): ptLasso và Trans-GLM trong E7, chạy lại trong kho survey

Notebook chạy bằng kernel **Python**: clone kho `RNA-Seq-FS-Survey` (nhánh `dmkd-revision`, commit đã push), ghi đầu vào bằng
`scripts/export_e7_r_inputs.py`, tự cài R riêng bằng micromamba (conda-forge; cách đã chạy được ở bài 2 ngày 06/11 và 09/11), gọi
`scripts/r/e7_r_baselines.R`, rồi `scripts/run_e7_r_eval.py` lấy panel và đánh giá. Mã ba tệp chép nguyên từ kho bài 2 tại `7f097da`.
Script R dừng (mã 2) nếu ptLasso khác 1.0.2 hoặc glmtrans khác 2.1.0.

| Bước | Việc | Kết quả cần xem |
|---|---|---|
| 1. Cổng | 5 lần chia đã công bố (seed 42–46), n_T ∈ {5, 10, 20}, k ∈ {2, 3, 4, 5, 10} → `B6_E7_gate5/`; so với `results/survey_v2/B6_reference_rq2_r_7f097da/` (trích `rq2_r_all_7f097da_results.zip`) bằng `make_revision_tables.py b6gate` | `B6_gate/b6_gate.csv`: `pass_all = True` cả 4 cohort |
| 2. 20 lần chia (`PHASE = 'all'`, chỉ khi cổng đạt) | bản chia B1 (seed 42–61, cổng lần chia 0–4) → `B1_splits/`; ptLasso, Trans-GLM → `B6_E7_principled/` | `e7_r_per_run.csv`, `e7_r_diagnostics.csv` |
| 3. Bảng (tuỳ chọn) | nếu `RESTORE_B1` trỏ tới output của JOB `b1` (`B1_E7_20/`): `make_revision_tables.py b6` → `B6_tables/` | nhãn cấp lần chia so với target-only |

Mỗi phiên:
1. Language: **Python**. Accelerator: None. Bật **Internet** (clone, tải R, tải gói từ CRAN).
2. Add Input: **experiment-data-v3**; (tuỳ chọn) output của JOB `b1` cho bước 3.
3. Ô cấu hình: `PHASE`, `REPO_COMMIT` (commit đã push), `INPUT_V3_DATASET`, `RESTORE_B1`.
4. Save Version → Save & Run All. Kết quả: `/kaggle/working/b6_<PHASE>_<commit7>_results.zip`.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, tarfile, time, urllib.request, zipfile

PHASE = 'all'                        # 'gate' (chỉ cổng 5 lần chia) hoặc 'all' (cổng rồi 20 lần chia)
REPO_URL = 'https://github.com/datachain-uit/RNA-Seq-FS-Survey.git'
REPO_BRANCH = 'dmkd-revision'
REPO_COMMIT = None                   # commit đã push chứa scripts/r/e7_r_baselines.R
INPUT_V3_DATASET = Path('/kaggle/input/experiment-data-v3')   # sửa theo đường dẫn trong panel Input
RESTORE_B1 = None                    # (tuỳ chọn) Input chứa b1_results.zip để làm bảng B6 ngay trong phiên
DATASETS = ['gse10245', 'gse30219', 'gse37745', 'gse42127']
N_T = ['5', '10', '20']              # lưới n_T của E7
PIP_PINS = {}                        # QĐ70: rỗng = giữ thư viện của image; nếu run_e7_r_eval.py dừng mã 2, dán PIP_PINS nó in ra

REPO_DIR = Path('/tmp/RNA-Seq-FS-Survey')
WORK_ROOT = Path('/kaggle/working/kltn_data')
RESULTS = WORK_ROOT / 'results' / 'survey_v2'
OUT_DIR = Path('/kaggle/working/b6')
LIB_DIR = Path('/tmp/r_baselines_lib')
MAMBA_DIR = Path('/tmp/micromamba')
R_ENV = Path('/tmp/r_env')
MICROMAMBA_URL = 'https://micro.mamba.pm/api/micromamba/linux-64/latest'
CONDA_PACKAGES = ['r-base==4.5.3', 'r-glmnet==5.0', 'r-ggplot2', 'r-gridextra', 'r-matrix', 'r-caret', 'r-foreach', 'r-doparallel', 'r-assertthat', 'r-formatr']
assert PHASE in ('gate', 'all'), PHASE
T0 = time.time()
print(sys.version)

In [ ]:
# Gate: clone đúng commit đã push; dữ liệu đúng layout; biến môi trường cho các script Python.
assert REPO_COMMIT, 'Điền REPO_COMMIT ở ô cấu hình'
if REPO_DIR.exists(): shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', '--quiet', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--quiet', REPO_COMMIT], cwd=REPO_DIR, check=True)
commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert subprocess.check_output(['git', 'branch', '-r', '--contains', 'HEAD'], cwd=REPO_DIR, text=True).strip(), f'{commit_hash} chưa có trên remote'
for f in ('scripts/r/e7_r_baselines.R', 'scripts/export_e7_r_inputs.py', 'scripts/run_e7_r_eval.py', 'scripts/make_revision_tables.py',
          'results/survey_v2/B6_reference_rq2_r_7f097da/SHA256SUMS.txt'):
    assert (REPO_DIR / f).is_file(), f'commit {commit_hash} không có {f}'
assert (INPUT_V3_DATASET / 'experiment-data-geo-target').is_dir(), f'Không thấy Kaggle Input: {INPUT_V3_DATASET}'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
for folder in ('experiment-data-perm', 'experiment-data-geo-target', 'split_manifests'):
    link = WORK_ROOT / folder
    if not link.exists(): link.symlink_to(INPUT_V3_DATASET / folder, target_is_directory=True)
if RESTORE_B1:
    for z in sorted(Path(RESTORE_B1).rglob('*_results.zip')):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(WORK_ROOT, members=[m for m in zf.namelist() if m.startswith('results/survey_v2/B1_E7_20/')])
        print('RESTORED', z)
if OUT_DIR.exists(): shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True)
os.environ.update({'KLTN_DATA_ROOT': str(WORK_ROOT), 'KLTN_STRICT_N': '1', 'PYTHONUTF8': '1'})
os.environ.pop('KLTN_ALLOW_UNPUSHED', None)
if PIP_PINS:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *[f'{p}=={v}' for p, v in PIP_PINS.items()]], check=True)
for mod in ('numpy', 'pandas', 'scipy', 'sklearn', 'tables'):
    print(f'{mod:10s}', getattr(__import__(mod), '__version__', 'ok'))
(OUT_DIR / 'pip_freeze.txt').write_text(subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout, encoding='utf-8')
print('COMMIT', commit_hash, '| PHASE', PHASE)

In [ ]:
# Cài R riêng bằng micromamba; lỗi được ghi lại, không dừng, để ô cuối vẫn lưu zip.
setup_error = None
micromamba = MAMBA_DIR / 'bin' / 'micromamba'
t0 = time.time()
try:
    MAMBA_DIR.mkdir(parents=True, exist_ok=True)
    archive_mm = MAMBA_DIR / 'micromamba.tar.bz2'
    urllib.request.urlretrieve(MICROMAMBA_URL, archive_mm)
    with tarfile.open(archive_mm) as tar:
        tar.extract('bin/micromamba', MAMBA_DIR)
    os.chmod(micromamba, 0o755)
    with open(OUT_DIR / 'setup_r.log', 'w', encoding='utf-8') as log:
        subprocess.run([str(micromamba), 'create', '--yes', '--quiet', '--prefix', str(R_ENV), '--override-channels', '--channel', 'conda-forge', *CONDA_PACKAGES],
                       stdout=log, stderr=subprocess.STDOUT, check=True, env={**os.environ, 'MAMBA_ROOT_PREFIX': str(MAMBA_DIR)})
    print(subprocess.check_output([str(R_ENV / 'bin' / 'Rscript'), '--version'], text=True, stderr=subprocess.STDOUT).strip())
except Exception as exc:
    setup_error = repr(exc)
    (OUT_DIR / 'setup_error.txt').write_text(setup_error, encoding='utf-8')
    print('LỖI CÀI R:', setup_error)
print(f'Cài R: {(time.time() - t0) / 60:.1f} phút')

In [ ]:
# Một lượt: export (Python) -> R -> panel và đánh giá (Python). Trả mã đầu tiên khác 0, log vào OUT_DIR/<tag>_*.log.
def run_py(tag, args):
    with open(OUT_DIR / f'{tag}.log', 'w', encoding='utf-8') as log:
        code = subprocess.run([sys.executable, *args], cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT).returncode
    print(tag, 'EXIT', code); print('\n'.join((OUT_DIR / f'{tag}.log').read_text(encoding='utf-8', errors='replace').splitlines()[-10:]))
    return code

def one_pass(tag, inputs_name, eval_out, split_args):
    inputs = RESULTS / inputs_name
    r_out = OUT_DIR / f'r_out_{tag}'
    code = run_py(f'{tag}_export', ['scripts/export_e7_r_inputs.py', '--datasets', *DATASETS, '--n-t', *N_T, *split_args, '--out', inputs_name])
    if code: return code
    env = {k: v for k, v in os.environ.items() if k not in ('LD_LIBRARY_PATH', 'R_HOME', 'PYTHONPATH')}
    env.update({'PATH': f"{R_ENV / 'bin'}:{env['PATH']}", 'R_BASELINE_LIB': str(LIB_DIR), 'REPO_COMMIT_HASH': commit_hash})
    t0 = time.time()
    with open(OUT_DIR / f'{tag}_r_stdout.log', 'w', encoding='utf-8') as out, open(OUT_DIR / f'{tag}_r_stderr.log', 'w', encoding='utf-8') as err:
        code = subprocess.run([str(R_ENV / 'bin' / 'Rscript'), 'scripts/r/e7_r_baselines.R', str(inputs), str(r_out), 'all', *DATASETS],
                              stdout=out, stderr=err, env=env, cwd=REPO_DIR).returncode
    print(tag, 'R EXIT', code, f'sau {(time.time() - t0) / 60:.1f} phút')
    print('\n'.join((OUT_DIR / f'{tag}_r_stdout.log').read_text(encoding='utf-8', errors='replace').splitlines()[-8:]))
    if code: return code
    return run_py(f'{tag}_eval', ['scripts/run_e7_r_eval.py', '--inputs', str(inputs), '--r-out', str(r_out), '--datasets', *DATASETS,
                                  '--save-scores', '--out', eval_out])

status = {}
if setup_error is None:
    status['gate_pass'] = one_pass('gate5', 'B6_r_inputs', 'B6_E7_gate5', [])
    if status['gate_pass'] == 0:
        status['gate_check'] = run_py('gate_check', ['scripts/make_revision_tables.py', 'b6gate'])
print(status)

In [ ]:
# 20 lần chia (chỉ khi cổng đạt): bản chia B1 (cổng lần chia 0–4), rồi cùng một lượt -> B6_E7_principled.
if PHASE == 'all' and status.get('gate_check') == 0:
    status['b1_splits'] = run_py('b1_splits', ['scripts/build_fewshot_splits.py', '--datasets', *DATASETS, '--seeds', *[str(s) for s in range(42, 62)],
                                               '--name', 'fewshot_splits_20.csv', '--out-root', str(RESULTS / 'B1_splits'),
                                               '--manifest-name', 'fewshot_splits_20_manifest.json', '--gate-against', 'fewshot_splits.csv'])
    if status['b1_splits'] == 0:
        status['all20'] = one_pass('all20', 'B6_r_inputs_20', 'B6_E7_principled',
                                   ['--splits-root', str(RESULTS / 'B1_splits'), '--splits-file', 'fewshot_splits_20.csv'])
    if status.get('all20') == 0 and (RESULTS / 'B1_E7_20').is_dir():
        status['tables'] = run_py('tables', ['scripts/make_revision_tables.py', 'b6'])
print(status)

In [ ]:
# Lưu kết quả rồi mới báo lỗi. Không đưa ma trận biểu hiện (*.csv.gz) vào zip; chúng tái tạo được từ commit + dữ liệu.
import json
stage_dir = OUT_DIR / 'results' / 'survey_v2'
for sub in ('B6_r_inputs', 'B6_r_inputs_20', 'B6_E7_gate5', 'B6_gate', 'B1_splits', 'B6_E7_principled', 'B6_tables'):
    src = RESULTS / sub
    if src.exists():
        shutil.copytree(src, stage_dir / sub, ignore=shutil.ignore_patterns('*.csv.gz'), dirs_exist_ok=True)
info = {'phase': PHASE, 'commit_hash': commit_hash, 'setup_error': setup_error, 'status': status,
        'hours': round((time.time() - T0) / 3600, 2), 'finished_at': time.strftime('%Y-%m-%d %H:%M:%S %Z')}
(OUT_DIR / 'run_info.json').write_text(json.dumps(info, indent=2), encoding='utf-8')
archive = shutil.make_archive(f'/kaggle/working/b6_{PHASE}_{commit_hash[:7]}_results', 'zip', root_dir=OUT_DIR.parent, base_dir=OUT_DIR.name)
print('ARCHIVE', archive, os.path.getsize(archive), 'byte', info)
if setup_error is not None: raise RuntimeError(f'Không cài được R: {setup_error}')
if status.get('gate_pass') != 0: raise RuntimeError('Lượt cổng lỗi (export / R / eval): xem gate5_*.log; mã 2 của eval = sai phiên bản thư viện (PIP_PINS)')
if status.get('gate_check') != 0: raise RuntimeError('CỔNG B6 KHÔNG ĐẠT: xem B6_gate/b6_gate.csv — không chạy 20 lần chia')
if PHASE == 'all' and status.get('all20') != 0: raise RuntimeError('Lượt 20 lần chia lỗi: xem b1_splits.log, all20_*.log')
print(f'B6_{PHASE.upper()}_OK')